# Deep Learning Assessment 1 Template

#Aim of Assessment 1

**You should 'Copy to Drive' this file so you have your own copy to modify.**


This is very similar to the code as you examined in Lab 3 (CNN).

Your key aim is to:

1. Do the four quiz questions for Assessment 1.
2. Develop a CNN model **with the same architecture as given in your quiz.**
3. Develop an equivalent fully-connected network with four fully connected layers which have (as close as possible) the same number of parameters as those in the four layers in your CNN network.
4. Calculate the average accuracy and standard deviation of both your models over 5 test runs (the random weight assignment will give different answers each time).
5. Work out the p-value (in code) for whether the CNN network is statically more accurate than your fully-connected network.

**DO NOT CHANGE ANY OTHER LINES OF THIS NOTEBOOK EXCEPT THOSE IN NOTEBOOK CELLS INDICATED**



In [ ]:
from google.colab import files

# Common imports
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import seaborn as sns

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize']  = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "pytorch"

# make sure you use the GPU (btw check your runtime is a GPU in colab)
use_cuda = True
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")


def save_fig(fig_id, tight_layout=True):
    path = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID, fig_id + ".png")
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format='png', dpi=300)
    files.download(PROJECT_ROOT_DIR+'/images/'+CHAPTER_ID+'/'+fig_id + ".png")

print("Using CUDA ? " + str(use_cuda))



# First load the data so you can see what the network needs to handle.

Now, let's train our ConvNet on the Fashion MNIST digits. You can[ learn more about the Fashion-MNIST data set](https://github.com/zalandoresearch/fashion-mnist).

We specify the root directory to store the dataset, download the training data, if not present on the local machine, and then apply the transforms. ToTensor to turn images into Tensor so we can directly use it with our network. The dataset is stored in the dataset class named `train_set` and the test data in (unsurprisingly) `test_set`.

In [ ]:
import torchvision
import torchvision.transforms as transforms

# Use standard FashionMNIST dataset
train_set = torchvision.datasets.FashionMNIST(
    root = './data/FashionMNIST',
    train = True,
    download = True,
    transform = transforms.Compose([
        transforms.ToTensor()
    ])
)

test_set = torchvision.datasets.FashionMNIST(
    root = './data/FashionMNIST',
    train = False,
    download = True,
    transform = transforms.Compose([
        transforms.ToTensor()
    ])
)
print(train_set, test_set)

# Set up a (train) loader and a test loader for providing the data.

In [ ]:
# set up a loader object to use minibatches
loader = torch.utils.data.DataLoader(train_set, batch_size = 32)

# also set up a loader for test data, using a single batch
test_loader = torch.utils.data.DataLoader(test_set)



This gets a single "mini-batch" of samples in the format that your model needs to handle. You should be able to pass these examples through your model successfully.

In [ ]:
images, labels = next(iter(loader))

print("Batch Input Shape = ",images.shape)
print("Batch Labels Shape = ",labels.shape)
print("")
print("Example Batch Input = ",images)
print("Example Batch Labels = ",labels)
print("")

# show the first batch of training data
grid = torchvision.utils.make_grid(images)
plt.imshow(np.transpose(grid.numpy(), (1,2,0)), interpolation='nearest')


**IMPORTANT: PUT THE CODE FOR YOUR MODELS ONLY IN THE NOTEBOOK CELLS INDICATED**

**DO NOT CHANGE ANY OF THE CODE OUTSIDE THE CELLS INDICATED!**

Fill in the missing structure to implement:

 1. A multi-layer convolutional network for `model_CNN` with an identical architecture as indicated in your quiz questions.

 2. A multi-layer linear, dense network for `model_dense` where the dense model has (as close as possible) the same number of parameters in each of its four layers as the number of parameters (in each layer) of your CNN model.


**IMPORTANT: You need to implement the CNN model architecture that is indicated in your quiz - you will loose marks otherwise!**

**IMPORTANT: You need to work out the number of parameters used by the different layers of your CNN network and design your fully-connected network to have the same number of parameters in each layer (as close as possible)**


In [ ]:
# **** YOU CAN MODIFY THIS NOTEBOOK CELL (8 marks) ***

# Please look up the nn.Sequential class in the PyTorch API to get help with this.

import collections

# CNN Model Architecture (matching quiz specifications)
# Input: 28x28x1 Fashion-MNIST images
# Layer 1: Conv2D with 3 kernels, 6x6 size, padding 1, stride 1, Tanh activation
# Layer 2: Conv2D with 5 kernels, 5x5 size, padding 2, stride 1, Sigmoid activation  
# Layer 3: Fully Connected with 67 neurons, ReLU activation
# Layer 4: Fully Connected with 10 neurons (output layer)

model_CNN = nn.Sequential(
    # Layer 1: Convolutional Layer
    # Input: 28x28x1 -> Output: 25x25x3
    # Parameters: (6*6*1 + 1) * 3 = 111
    nn.Conv2d(in_channels=1, out_channels=3, kernel_size=6, stride=1, padding=1, bias=True),
    nn.Tanh(),
    
    # Layer 2: Convolutional Layer  
    # Input: 25x25x3 -> Output: 25x25x5
    # Parameters: (5*5*3 + 1) * 5 = 380
    nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5, stride=1, padding=2, bias=True),
    nn.Sigmoid(),
    
    # Flatten the output for fully connected layers
    # Input: 25x25x5 -> Output: 3125
    nn.Flatten(),
    
    # Layer 3: Fully Connected Layer
    # Input: 3125 -> Output: 67
    # Parameters: 3125*67 + 67 = 209,442 (including 67 bias parameters)
    nn.Linear(in_features=3125, out_features=67, bias=True),
    nn.ReLU(),
    
    # Layer 4: Fully Connected Output Layer
    # Input: 67 -> Output: 10
    # Parameters: 67*10 + 10 = 680
    nn.Linear(in_features=67, out_features=10, bias=True)
)

# Fully-Connected Model (matching parameter counts as closely as possible)
# Total CNN parameters: 111 + 380 + 209,442 + 680 = 210,613
# Dense network designed to have similar total parameters: ~210,977
# Architecture: 784 -> 222 -> 124 -> 67 -> 10

model_dense = nn.Sequential(
    # Flatten the input images
    # Input: 28x28x1 -> Output: 784
    nn.Flatten(),
    
    # Layer 1: Fully Connected
    # Input: 784 -> Output: 222
    # Parameters: 784*222 + 222 = 174,270
    nn.Linear(in_features=784, out_features=222, bias=True),
    nn.Tanh(),
    
    # Layer 2: Fully Connected
    # Input: 222 -> Output: 124  
    # Parameters: 222*124 + 124 = 27,652
    nn.Linear(in_features=222, out_features=124, bias=True),
    nn.Sigmoid(),
    
    # Layer 3: Fully Connected
    # Input: 124 -> Output: 67
    # Parameters: 124*67 + 67 = 8,375
    nn.Linear(in_features=124, out_features=67, bias=True),
    nn.ReLU(),
    
    # Layer 4: Fully Connected Output Layer
    # Input: 67 -> Output: 10
    # Parameters: 67*10 + 10 = 680 (matches CNN exactly)
    nn.Linear(in_features=67, out_features=10, bias=True)
)

Now here's what the architecture of our two networks that you have developed looks like:

In [ ]:

print("\nCNN Network Model:")
print(model_CNN)

print("Dense MLP Model:")
print(model_dense)


In [ ]:
#
# TRY OUT YOUR MODEL HERE USING THE EXAMPLE BATCH INPUT ABOVE.
# (This will allow you to fix issues before running the training loop.)
# If this produces an error then you probably haven't defined your dense model correctly.
#

model_CNN_output = model_CNN(images)

print("Model output = ", model_CNN_output)
print("Ground Truth batch output = ", labels)

# Is the output from the model what you expected ?


In [ ]:
#
# TRY OUT YOUR MODEL HERE USING THE EXAMPLE BATCH INPUT ABOVE.
# (This will allow you to fix issues before running the training loop.)
# If this produces an error then you probably haven't defined your dense model correctly.
#

model_dense_output = model_dense(images)

print("Model output = ", model_dense_output)
print("Ground Truth batch output = ", labels)

# Is the output from the model what you expected ?


Then we need to define functions for training and testing our networks, the training loop here is fairly standard and is very much the same as in previous weeks.

In [ ]:
import datetime
epoch_print_gap = 1

def training_loop(n_epochs, optimizer, model, device, loss_fn, train_loader):
    model = model.to(device)
    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            outputs = model(imgs.to(device))
            loss = loss_fn(outputs, labels.to(device))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item()

        if epoch == 1 or epoch % epoch_print_gap == 0:
            print('{} Epoch {}, Training loss {}'.format(
                datetime.datetime.now(), epoch, float(loss_train)))

def test_loop(model, device, test_loader):
    model.eval()
    model = model.to(device)
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),accuracy))

    return accuracy


We then initialise the parameters for the training of our networks: optimiser (Stochastic Gradient Descent here), and data loader to pack your training data in mini-batches.

In [ ]:
lr = 0.01


# we set the optimiser as Stochastic Gradient Descent for both networks,
optimizer_CNN = optim.SGD(model_CNN.parameters(), lr=lr)
optimizer_dense = optim.SGD(model_dense.parameters(), lr=lr)

# set the loss function to optimise. Cross Entropy is usually the best for
# classification
loss_fn = nn.CrossEntropyLoss()

Finally, we can call the training loop on both models to see how they train. It is a good idea here to ensure some output in your training function so that you get a feeling how fast (and how well) the training goes and that you can abort a training that would be unreasonably long...

In [ ]:

# set the number of epochs: The number of time that the whole training set
# will be seen by the network during training
n_epochs = 2 # see how sensitive results are to this


# train the CNN
training_loop(
    n_epochs = n_epochs,
    optimizer = optimizer_CNN,
    model = model_CNN,
    device = device,
    loss_fn = loss_fn,
    train_loader = loader,
)


# train the dense model
training_loop(
    n_epochs = n_epochs,
    optimizer = optimizer_dense,
    model = model_dense,
    device = device,
    loss_fn = loss_fn,
    train_loader = loader,
)



We can then run the test loop using the two different models to assess performance.

In [ ]:
test_loop(model = model_CNN, device = device, test_loader = test_loader)
test_loop(model = model_dense, device = device, test_loader = test_loader)

In the following notebook cell, your aim is to do a computational experiment to find out whether the CNN architecture is generally more accurate than a fully-connected architecture for image recognition.

**Using a loop,** write code in the next notebook cell to **train and run 5 different models (of each type)** and work out the mean accuracy of each model and its standard deviation.
(Each trained model should be different due to random weight initialization.)

Also calculate the p-value for the CNN model being better than the fully connected model.

In [ ]:
# **** YOU CAN MODIFY THIS NOTEBOOK CELL (4 marks) ***

# ** Using a loop **, calculate the mean and standard deviation of accuracy over 5 copies of each model (and print these out).

# Calculate the p-value (and print out) for whether the CNN model is statically significantly more accurate than the fully connected model.

from scipy import stats

# Number of runs for statistical analysis
n_runs = 5

# Lists to store accuracy results
cnn_accuracies = []
dense_accuracies = []

print("=" * 70)
print("RUNNING 5 TRIALS FOR STATISTICAL ANALYSIS")
print("=" * 70)

# Run 5 trials with different random initializations
for run in range(1, n_runs + 1):
    print(f"\n{'='*70}")
    print(f"TRIAL {run}/{n_runs}")
    print(f"{'='*70}\n")
    
    # Reinitialize models (this creates new random weights)
    # CNN Model
    model_CNN_trial = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=3, kernel_size=6, stride=1, padding=1, bias=True),
        nn.Tanh(),
        nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5, stride=1, padding=2, bias=True),
        nn.Sigmoid(),
        nn.Flatten(),
        nn.Linear(in_features=3125, out_features=67, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=67, out_features=10, bias=True)
    )
    
    # Dense Model  
    model_dense_trial = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=784, out_features=222, bias=True),
        nn.Tanh(),
        nn.Linear(in_features=222, out_features=124, bias=True),
        nn.Sigmoid(),
        nn.Linear(in_features=124, out_features=67, bias=True),
        nn.ReLU(),
        nn.Linear(in_features=67, out_features=10, bias=True)
    )
    
    # Create new optimizers for this trial
    optimizer_CNN_trial = optim.SGD(model_CNN_trial.parameters(), lr=lr)
    optimizer_dense_trial = optim.SGD(model_dense_trial.parameters(), lr=lr)
    
    # Train CNN model
    print(f"Training CNN model (Trial {run})...")
    training_loop(
        n_epochs=n_epochs,
        optimizer=optimizer_CNN_trial,
        model=model_CNN_trial,
        device=device,
        loss_fn=loss_fn,
        train_loader=loader
    )
    
    # Train Dense model
    print(f"\nTraining Dense model (Trial {run})...")
    training_loop(
        n_epochs=n_epochs,
        optimizer=optimizer_dense_trial,
        model=model_dense_trial,
        device=device,
        loss_fn=loss_fn,
        train_loader=loader
    )
    
    # Test CNN model
    print(f"\nTesting CNN model (Trial {run})...")
    cnn_acc = test_loop(model=model_CNN_trial, device=device, test_loader=test_loader)
    cnn_accuracies.append(cnn_acc)
    
    # Test Dense model
    print(f"Testing Dense model (Trial {run})...")
    dense_acc = test_loop(model=model_dense_trial, device=device, test_loader=test_loader)
    dense_accuracies.append(dense_acc)
    
    print(f"\nTrial {run} Results: CNN={cnn_acc:.2f}%, Dense={dense_acc:.2f}%")

# Convert to numpy arrays for easier computation
cnn_accuracies = np.array(cnn_accuracies)
dense_accuracies = np.array(dense_accuracies)

# Calculate statistics
cnn_mean = np.mean(cnn_accuracies)
cnn_std = np.std(cnn_accuracies, ddof=1)  # Sample standard deviation

dense_mean = np.mean(dense_accuracies)
dense_std = np.std(dense_accuracies, ddof=1)  # Sample standard deviation

# Perform one-tailed paired t-test
# H0: CNN accuracy <= Dense accuracy
# H1: CNN accuracy > Dense accuracy (one-tailed test)
t_statistic, p_value_two_tailed = stats.ttest_rel(cnn_accuracies, dense_accuracies)

# Convert to one-tailed p-value (testing if CNN > Dense)
if t_statistic > 0:  # CNN mean is greater than Dense mean
    p_value = p_value_two_tailed / 2
else:  # CNN mean is less than or equal to Dense mean
    p_value = 1 - (p_value_two_tailed / 2)

# Print results
print("\n" + "=" * 70)
print("STATISTICAL ANALYSIS RESULTS")
print("=" * 70)
print(f"\nCNN Model Accuracy:")
print(f"  Individual trials: {cnn_accuracies}")
print(f"  Mean Accuracy: {cnn_mean:.2f}%")
print(f"  Standard Deviation: {cnn_std:.2f}%")

print(f"\nDense Model Accuracy:")
print(f"  Individual trials: {dense_accuracies}")
print(f"  Mean Accuracy: {dense_mean:.2f}%")
print(f"  Standard Deviation: {dense_std:.2f}%")

print(f"\nComparison:")
print(f"  Accuracy Difference (CNN - Dense): {cnn_mean - dense_mean:.2f}%")

print(f"\nStatistical Significance Test:")
print(f"  Test: One-tailed paired t-test")
print(f"  Null Hypothesis (H0): CNN accuracy <= Dense accuracy")
print(f"  Alternative Hypothesis (H1): CNN accuracy > Dense accuracy")
print(f"  t-statistic: {t_statistic:.4f}")
print(f"  p-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"\n  Result: The CNN model is statistically significantly more accurate")
    print(f"          than the Dense model (p < 0.05)")
else:
    print(f"\n  Result: No statistically significant difference found (p >= 0.05)")
    
print("=" * 70)

**IMPORTANT - Once you have completed this notebook:**

**1. Ensure that all the output of the notebook cells are all showing**
** (potentially by doing 'Runtime>Run All' to ensure everything runs ok and produces all the output).**

**2. Download a copy of your notebook using 'File > Download > Download .ipynb'.**

**3. Upload your notebook to Moodle under Assessment 1.**
